# Dr Sunil Godara — Credit-free Voice Refinement Lab

Run this notebook in Google Colab. It uses Qwen3-TTS for experimental voice-cloning and pronunciation QA. Your reference WAV is uploaded privately to the temporary Colab runtime and is **not stored in this repository**.

**Privacy:** do not commit or upload the reference recording to GitHub. Delete the Colab runtime when finished if you do not need its files.

**Commercial use:** the Qwen3-TTS project/model cards declare Apache-2.0 licensing. Check the current model card, dependencies and licence notices before commercial deployment. This is not legal advice. Free Colab GPU availability is not guaranteed.


In [ ]:
!pip -q install -U qwen-tts soundfile


In [ ]:
import torch, soundfile as sf, os
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU allocated. In Colab choose Runtime → Change runtime type → T4 GPU, if available, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Upload the authorized reference recording
Upload the WAV file `1000133293_audio.wav` previously supplied for this project. It stays in the Colab runtime; it is not part of this GitHub repository.


In [ ]:
from google.colab import files
uploaded = files.upload()
wav_candidates = [n for n in uploaded if n.lower().endswith('.wav')]
if not wav_candidates: raise ValueError('Please upload the authorized reference WAV.')
ref_audio = wav_candidates[0]
info = sf.info(ref_audio)
print('Reference:', ref_audio, '| Duration:', round(info.duration,1), 'sec | Sample rate:', info.samplerate, '| Channels:', info.channels)


## 2. Load Qwen3-TTS Base
The model weights are downloaded into this temporary Colab runtime.


In [ ]:
from qwen_tts import Qwen3TTSModel
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-0.6B-Base', device_map='cuda:0', dtype=dtype)
print('Model loaded.')


## 3. Generate the four pronunciation tests
This initial pass uses speaker-embedding-only cloning, so it does not require a transcript of the reference recording.


In [ ]:
sentences = [
    'I sourced this gimbal from Kazakhstan.',
    'Sourcing is always the most difficult part.',
    'No matter where you source from, we source the best.',
    'No middleman, source from source.'
]
wavs, sample_rate = model.generate_voice_clone(text=sentences, language=['English'] * len(sentences), ref_audio=ref_audio, x_vector_only_mode=True)
os.makedirs('voice_tests', exist_ok=True)
for i, (sentence, audio) in enumerate(zip(sentences, wavs), start=1):
    out = f'voice_tests/test_{i}.wav'
    sf.write(out, audio, sample_rate)
    print(i, sentence, '→', out)


## 4. Listen and download
Listen for a clear but natural final /r/ in *source*, and fluent connected speech in *sourced* and *sourcing*.


In [ ]:
from IPython.display import Audio, display
for i in range(1, 5):
    print(sentences[i-1])
    display(Audio(f'voice_tests/test_{i}.wav'))


In [ ]:
!zip -qr Dr_Sunil_voice_tests.zip voice_tests
files.download('Dr_Sunil_voice_tests.zip')
